# Goal

This notebook explains the foundation of our ViT + SAE project in a compact, implementation-oriented way.

It answers these questions:

0. Overall pipeline
1. What is SAE and how is it implemented?
2. What is ViT-B/16 and what are its inputs/outputs?
3. Which ViT layers should we analyze and why?
4. Which dataset should we use?
5. What preprocessing is required before feeding images into ViT?
6. What activation format will be passed to the SAE training pipeline?

# Key Observations / Decisions

This notebook is now sufficient to answer the foundation questions for the ViT + SAE project. The short answers are:

1. **SAE implementation**: the SAE is a separate two-layer model trained on frozen ViT activations. It uses `Linear(768 -> d_sae)`, `ReLU`, and `Linear(d_sae -> 768)`, with loss `MSE reconstruction + L1 sparsity`.
2. **ViT-B/16 understanding**: ViT-B/16 takes `[batch, 3, 224, 224]` images, splits them into `196` patches plus one CLS token, produces token activations shaped `[batch, 197, 768]`, and returns ImageNet logits shaped `[batch, 1000]`. We use `ViT_B_16_Weights.IMAGENET1K_V1`.
3. **Layer decision**: for patch-token SAE, early layers such as 3/4 are best for low-level local visual features, but **layer 9** is the best current choice for semantic local-region/object-related patch features. Layer 6 is a middle baseline, and layer 11 is only a late/global comparison.
4. **Dataset decision**: use the local **ImageNet-100 validation set** now: `100 classes x 50 images = 5,000 images`. Use 500-image subsets for layer-selection probes. Full ImageNet-1K train is too large for this stage; ImageNet-1K validation is a future scaling option.
5. **Preprocessing decision**: preprocessing is required, but it should come from `weights.transforms()` rather than hand-written resize/crop/normalize code. The official transform resizes, center-crops to `224 x 224`, converts to tensor, and normalizes with ImageNet mean/std.


# 0. Overall Pipeline

The full workflow starts with raw images, extracts internal ViT activations, trains an SAE on those activation vectors, and then ranks/visualizes SAE features for interpretation.

![Overall ViT + SAE pipeline](../images/overall_pipeline.png)

# 1. What is SAE and how is it implemented?

A sparse autoencoder (SAE) is a small neural network trained to reconstruct another model's activations using a sparse hidden representation.

In this project, the ViT is frozen. We first collect ViT activation vectors, then train a separate SAE on those vectors.

For one ViT activation vector `x` with dimension `768`, the SAE does:

```text
x -> encoder -> sparse code z -> decoder -> reconstructed activation x'
```

The hidden code `z` is larger than the input dimension, but the L1 penalty encourages only a few SAE features to be active for each input.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=768, d_sae=4096):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z


sae = SparseAutoencoder(d_in=768, d_sae=4096)
sae

SparseAutoencoder(
  (encoder): Linear(in_features=768, out_features=4096, bias=True)
  (decoder): Linear(in_features=4096, out_features=768, bias=True)
)

The training objective has two parts:

1. **Reconstruction loss**: make `x'` close to the original ViT activation `x`.
2. **Sparsity loss**: make the SAE code `z` sparse, so each input uses only a small number of features.

```text
total_loss = MSE(x_hat, x) + l1_coeff * mean(abs(z))
```

After training, each dimension of `z` is treated as one SAE feature. To interpret a feature, we find the images or patches where that feature has the highest activation.

In [2]:
batch_size = 8
d_in = 768
l1_coeff = 3e-3

# Example batch of ViT activation vectors.
# In the real pipeline, these come from hooked ViT layer activations.
x = torch.randn(batch_size, d_in)

x_hat, z = sae(x)

reconstruction_loss = F.mse_loss(x_hat, x)
sparsity_loss = z.abs().mean()
total_loss = reconstruction_loss + l1_coeff * sparsity_loss

print(f"input shape:          {tuple(x.shape)}")
print(f"sparse code shape:    {tuple(z.shape)}")
print(f"reconstruction shape: {tuple(x_hat.shape)}")
print(f"reconstruction loss:  {reconstruction_loss.item():.4f}")
print(f"sparsity loss:        {sparsity_loss.item():.4f}")
print(f"total loss:           {total_loss.item():.4f}")

input shape:          (8, 768)
sparse code shape:    (8, 4096)
reconstruction shape: (8, 768)
reconstruction loss:  1.0737
sparsity loss:        0.2315
total loss:           1.0744


In our ViT SAE experiments, the same implementation is used with different activation sources:

- **Patch-token SAE**: input shape is `[num_images * 196, 768]`; each input is one image patch.
- **CLS-token SAE**: input shape is `[num_images, 768]`; each input summarizes one full image.

The SAE itself does not know whether the vector came from a patch token or a CLS token. It only learns to reconstruct 768-dimensional ViT activation vectors sparsely.

# 2. Understand ViT-B/16

ViT-B/16 means **Vision Transformer Base with 16 x 16 image patches**.

Its original goal is image classification. In this project, we use it for a different purpose: we freeze the pretrained ViT and use its internal activations as data for SAE training.

Key facts:

- **Required input**: RGB image tensor with shape `[batch, 3, 224, 224]`.
- **Patch size**: `16 x 16` pixels.
- **Number of patches**: `14 x 14 = 196` patches for a `224 x 224` image.
- **Token sequence**: `1 CLS token + 196 patch tokens = 197 tokens`.
- **Hidden dimension**: each token is represented by a `768`-dimensional vector.
- **Transformer layers**: 12 encoder blocks.
- **Attention heads**: 12 heads.
- **Classification output**: logits with shape `[batch, 1000]` for ImageNet-1K classes.
- **Weights used here**: `ViT_B_16_Weights.IMAGENET1K_V1`.

For SAE analysis, the most important returned object is not the final classifier output. Instead, we use forward hooks to collect selected layer activations with shape `[batch, 197, 768]`.

ViT-B/16 uses 12 transformer encoder blocks and 12 attention heads because these are the standard architecture settings of the ViT-Base model.
Each block updates the token representations through self-attention and an MLP.
The 12 attention heads split the 768-dimensional token representation into 12 smaller attention subspaces of 64 dimensions each.

The final output of ViT-B/16 is a logits vector with shape `[batch, 1000]`, where each value is an unnormalized score for one ImageNet-1K class.
Logits are not probabilities. They can be converted into probabilities using softmax. In this project, however, we do not use the final logits for SAE training. Instead, we use forward hooks to collect intermediate activations with shape `[batch, 197, 768]`.


In [3]:
from torchvision.models import vit_b_16, ViT_B_16_Weights

# Official pretrained ImageNet-1K weights and matching preprocessing.
weights = ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

# This loads the pretrained ViT-B/16 model.
# If the weights are not cached locally, torchvision may download them.
model = vit_b_16(weights=weights)
model.eval()

print("Weights:", weights)
print("Number of ImageNet categories:", len(weights.meta["categories"]))
print("Preprocessing:", preprocess)
hidden_dim = model.hidden_dim if hasattr(model, "hidden_dim") else model.conv_proj.out_channels
num_heads = model.encoder.layers[0].self_attention.num_heads

print("Patch size:", model.patch_size)
print("Hidden dimension:", hidden_dim)
print("Number of encoder layers:", len(model.encoder.layers))
print("Number of attention heads:", num_heads)
print("Classifier head:", model.heads)

# Example model input and output shapes.
example_x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    logits = model(example_x)

print("Input shape:", tuple(example_x.shape))
print("Returned logits shape:", tuple(logits.shape))

Weights: ViT_B_16_Weights.IMAGENET1K_V1
Number of ImageNet categories: 1000
Preprocessing: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)
Patch size: 16
Hidden dimension: 768
Number of encoder layers: 12
Number of attention heads: 12
Classifier head: Sequential(
  (head): Linear(in_features=768, out_features=1000, bias=True)
)
Input shape: (2, 3, 224, 224)
Returned logits shape: (2, 1000)


In [4]:
# A forward hook lets us inspect the internal token activations.
activations = {} # dic
target_layer = model.encoder.layers[5]  # layer 6 in human-readable numbering


def save_layer_activation(module, inputs, output):
    activations["layer_6"] = output.detach() # only save values


hook_handle = target_layer.register_forward_hook(save_layer_activation)

with torch.no_grad():
    _ = model(example_x)

hook_handle.remove()

layer_activation = activations["layer_6"]
cls_tokens = layer_activation[:, 0, :]
patch_tokens = layer_activation[:, 1:, :]

print("Layer activation shape:", tuple(layer_activation.shape))
print("CLS token shape:", tuple(cls_tokens.shape))
print("Patch tokens shape:", tuple(patch_tokens.shape))

assert list(layer_activation.shape) == [2, 197, 768]
assert list(cls_tokens.shape) == [2, 768]
assert list(patch_tokens.shape) == [2, 196, 768]

Layer activation shape: (2, 197, 768)
CLS token shape: (2, 768)
Patch tokens shape: (2, 196, 768)


# 3. SAE-Based Layer Selection

The layer choice should be based on how well an SAE works on activations from that layer, not only on raw ViT activation statistics.

For this project, we use four candidate layers: **3, 6, 9, and 11**. They give a simple depth sweep through ViT-B/16:

- **Layer 3**: early representation; useful as a low-level visual baseline for color, edge, and texture features.
- **Layer 6**: middle representation; likely to preserve local patch structure while becoming more abstract than early layers.
- **Layer 9**: late-middle representation; useful for more semantic patch features and object-region behavior.
- **Layer 11**: late representation; closest to classification behavior, useful as a semantic/late-layer comparison.

The goal of this section is not to train the final SAE. Instead, we train a small SAE for each candidate layer using the same setup, then compare basic SAE quality metrics.

Selection metrics:

- `normalized_reconstruction_mse`: lower is better; the SAE reconstructs the layer activation more easily.
- `l0_mean`: average number of active SAE features per patch activation; this checks sparsity.
- `active_feature_fraction` and `dead_feature_fraction`: check whether the SAE dictionary is being used.
- `avg_top10_class_purity`: for active SAE features, whether their top activating patches come from coherent classes.
- `avg_top10_unique_images`: checks whether top activations are not all from one image.
- `sae_layer_score`: a lightweight heuristic combining reconstruction, feature use, sparsity, and top-patch coherence.

This is the right kind of probe for SAE layer selection because it evaluates the actual SAE behavior on each candidate layer.


In [5]:
from pathlib import Path
from collections import Counter
from PIL import Image
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights


project_root = Path.cwd()
if not (project_root / "outputs").exists():
    project_root = project_root.parent

candidate_layers = [3, 6, 9, 11]
num_classes = 10
images_per_class = 4
image_batch_size = 8
sae_batch_size = 1024
d_sae = 1024
num_epochs = 10
l1_coeff = 5e-2
learning_rate = 1e-3

weights = ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

# Load the pretrained ViT if it is not already available from section 2.
if "model" not in globals():
    model = vit_b_16(weights=weights)
    model.eval()

subset_path = project_root / "outputs" / "activations" / "diverse_subset_200.csv"
subset = pd.read_csv(subset_path)
probe_subset = (
    subset.groupby("class_id", group_keys=False)
    .head(images_per_class)
    .head(num_classes * images_per_class)
    .reset_index(drop=True)
)

layer_activations = {layer: [] for layer in candidate_layers}
hook_handles = []


def make_hook(layer_number):
    def save_activation(module, inputs, output):
        layer_activations[layer_number].append(output.detach().cpu())
    return save_activation


for layer_number in candidate_layers:
    layer_index = layer_number - 1
    handle = model.encoder.layers[layer_index].register_forward_hook(make_hook(layer_number))
    hook_handles.append(handle)

for start in range(0, len(probe_subset), image_batch_size):
    batch_rows = probe_subset.iloc[start:start + image_batch_size]
    batch_images = [
        preprocess(Image.open(image_path).convert("RGB"))
        for image_path in batch_rows["image_path"]
    ]
    batch_x = torch.stack(batch_images, dim=0)

    with torch.no_grad():
        _ = model(batch_x)

for handle in hook_handles:
    handle.remove()


class ProbeSAE(nn.Module):
    def __init__(self, d_in=768, d_sae=1024):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z


flat_class_ids = []
flat_image_indices = []
for image_index, image_row in probe_subset.iterrows():
    for _ in range(196):
        flat_class_ids.append(image_row["class_id"])
        flat_image_indices.append(image_index)

result_rows = []

def top_patch_feature_stats(z, activation_frequency, max_activation, top_features=20, top_k=10):
    candidate_feature_ids = torch.where(
        (activation_frequency >= 0.005) & (activation_frequency <= 0.5)
    )[0]
    if len(candidate_feature_ids) == 0:
        candidate_feature_ids = torch.arange(z.shape[1])

    candidate_feature_ids = candidate_feature_ids[
        torch.argsort(max_activation[candidate_feature_ids], descending=True)[:top_features]
    ]

    purities = []
    unique_image_counts = []

    for feature_id in candidate_feature_ids.tolist():
        top_indices = torch.topk(z[:, feature_id], k=top_k).indices.tolist()
        top_classes = [flat_class_ids[index] for index in top_indices]
        top_images = [flat_image_indices[index] for index in top_indices]

        dominant_class_count = Counter(top_classes).most_common(1)[0][1]
        purities.append(dominant_class_count / top_k)
        unique_image_counts.append(len(set(top_images)))

    return {
        "avg_top10_class_purity": sum(purities) / len(purities),
        "avg_top10_unique_images": sum(unique_image_counts) / len(unique_image_counts),
    }


for layer_number in candidate_layers:
    torch.manual_seed(42)

    patch_tokens = torch.cat(layer_activations[layer_number], dim=0)[:, 1:, :].float()
    acts = patch_tokens.reshape(-1, 768)
    input_variance = acts.var(dim=0).mean().item()

    sae = ProbeSAE(d_in=768, d_sae=d_sae)
    optimizer = torch.optim.Adam(sae.parameters(), lr=learning_rate)
    dataloader = DataLoader(TensorDataset(acts), batch_size=sae_batch_size, shuffle=True)

    for _ in range(num_epochs):
        for (batch_acts,) in dataloader:
            x_hat, z = sae(batch_acts)
            reconstruction_loss = F.mse_loss(x_hat, batch_acts)
            sparsity_loss = z.abs().mean()
            total_loss = reconstruction_loss + l1_coeff * sparsity_loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

    sae.eval()
    with torch.no_grad():
        x_hat, z = sae(acts)
        reconstruction_mse = F.mse_loss(x_hat, acts).item()
        normalized_reconstruction_mse = reconstruction_mse / max(input_variance, 1e-8)
        active_mask = z > 1e-6
        l0_mean = active_mask.float().sum(dim=1).mean().item()
        activation_frequency = active_mask.float().mean(dim=0)
        max_activation = z.max(dim=0).values
        active_feature_fraction = (activation_frequency > 0.001).float().mean().item()
        dead_feature_fraction = (activation_frequency == 0).float().mean().item()

    feature_stats = top_patch_feature_stats(z, activation_frequency, max_activation)
    feature_diversity_factor = feature_stats["avg_top10_unique_images"] / 10
    sparsity_factor = max(0.0, 1.0 - min(l0_mean / d_sae, 1.0))

    sae_layer_score = (
        feature_stats["avg_top10_class_purity"]
        * active_feature_fraction
        * feature_diversity_factor
        * (0.25 + sparsity_factor)
        / max(normalized_reconstruction_mse, 1e-8)
    )

    result_rows.append({
        "layer": layer_number,
        "activation_shape": tuple(patch_tokens.shape),
        "flattened_shape": tuple(acts.shape),
        "d_sae": d_sae,
        "num_epochs": num_epochs,
        "l1_coeff": l1_coeff,
        "reconstruction_mse": reconstruction_mse,
        "normalized_reconstruction_mse": normalized_reconstruction_mse,
        "l0_mean": l0_mean,
        "active_feature_fraction": active_feature_fraction,
        "dead_feature_fraction": dead_feature_fraction,
        **feature_stats,
        "sae_layer_score": sae_layer_score,
    })

sae_layer_selection = pd.DataFrame(result_rows).sort_values(
    "sae_layer_score",
    ascending=False,
).reset_index(drop=True)

output_path = project_root / "outputs" / "feature_rankings" / "sae_layer_selection_probe_layers_3_6_9_11.csv"
sae_layer_selection.to_csv(output_path, index=False)

print(f"Probe subset: {len(probe_subset)} images from {probe_subset['class_id'].nunique()} classes")
print(f"Saved SAE layer-selection probe: {output_path}")
display(sae_layer_selection)



Probe subset: 40 images from 10 classes
Saved SAE layer-selection probe: /Users/queen/PycharmProjects/VITSAE/outputs/feature_rankings/sae_layer_selection_probe_layers_3_6_9_11.csv


,layer,activation_shape,flattened_shape,d_sae,num_epochs,l1_coeff,reconstruction_mse,normalized_reconstruction_mse,l0_mean,active_feature_fraction,dead_feature_fraction,avg_top10_class_purity,avg_top10_unique_images,sae_layer_score
0,6,"(40, 196, 768)","(7840, 768)",1024,10,0.05,0.023897,0.086986,589.778198,1.000000,0.000000,0.425,8.60,2.832230
1,3,"(40, 196, 768)","(7840, 768)",1024,10,0.05,0.006093,0.088985,521.940552,1.000000,0.000000,0.455,5.70,2.157608
2,9,"(40, 196, 768)","(7840, 768)",1024,10,0.05,0.049440,0.116870,623.473206,1.000000,0.000000,0.410,8.70,1.956828
3,11,"(40, 196, 768)","(7840, 768)",1024,10,0.05,0.112173,0.171723,387.433685,0.948242,0.005859,0.415,8.65,1.727804


The lightweight SAE probe result is saved to:

```text
outputs/feature_rankings/sae_layer_selection_probe_layers_3_6_9_11.csv
```

This first probe uses only `10 classes x 4 images = 40 images`, so it should be treated as a **smoke test**, not as the final layer decision.

Observed 40-image smoke-test result:

| rank | layer | normalized reconstruction MSE | L0 mean | active feature fraction | dead feature fraction | avg top-10 class purity | avg top-10 unique images | SAE layer score |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 6 | 0.0870 | 589.78 | 1.0000 | 0.0000 | 0.425 | 8.60 | 2.8322 |
| 2 | 3 | 0.0890 | 521.94 | 1.0000 | 0.0000 | 0.455 | 5.70 | 2.1576 |
| 3 | 9 | 0.1169 | 623.47 | 1.0000 | 0.0000 | 0.410 | 8.70 | 1.9568 |
| 4 | 11 | 0.1717 | 387.43 | 0.9482 | 0.0059 | 0.415 | 8.65 | 1.7278 |

Interpretation of this small probe:

- Layer 6 ranks highest on this tiny subset, so it is a useful middle-layer candidate.
- Layer 3 is a low-level baseline candidate.
- Layer 9 remains a semantic patch candidate, but this 40-image run is too small to judge it reliably.
- Layer 11 is late and more global/classification-oriented, so it is not a strong main patch-token layer.

Important update after larger experiments:

- This 40-image result is **superseded** by the later 500-image and all-12-layer probes.
- Do not use this table alone to choose the final layer.
- The final recommendation is made after the all-layer trend analysis below.


# 3b. Larger SAE Layer-Selection Probe: 50 Classes x 10 Images

The previous probe uses `10 classes x 4 images = 40 images`, which is useful as a smoke test.

For a more convincing layer-selection step, we run the same SAE-based comparison on a larger subset:

```text
50 classes x 10 images = 500 images
500 images x 196 patch tokens = 98,000 SAE training vectors per layer
```

This is still lightweight compared with a full SAE training run, but it is much less sensitive to one-off images or classes.

To keep memory usage controlled, this code processes one candidate layer at a time: extract patch activations, train a small SAE, record metrics, then move to the next layer.


In [6]:
from pathlib import Path
from collections import Counter
from PIL import Image
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights
from IPython.display import display


project_root = Path.cwd()
if not (project_root / "outputs").exists():
    project_root = project_root.parent

candidate_layers = [3, 6, 9, 11]
num_classes = 50
images_per_class = 10
image_batch_size = 16
sae_batch_size = 2048
d_sae = 1024
num_epochs = 5
l1_coeff = 5e-2
learning_rate = 1e-3

weights = ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

if "model" not in globals():
    model = vit_b_16(weights=weights)
    model.eval()

data_root = project_root / "data" / "val.X"
class_dirs = sorted([path for path in data_root.iterdir() if path.is_dir()])
eligible_class_dirs = [
    class_dir for class_dir in class_dirs
    if len(list(class_dir.glob("*.JPEG"))) >= images_per_class
]
selected_class_dirs = eligible_class_dirs[:num_classes]

subset_rows = []
for class_index, class_dir in enumerate(selected_class_dirs):
    for image_path in sorted(class_dir.glob("*.JPEG"))[:images_per_class]:
        subset_rows.append({
            "image_index": len(subset_rows),
            "class_index": class_index,
            "class_id": class_dir.name,
            "image_path": str(image_path),
        })

large_probe_subset = pd.DataFrame(subset_rows)
large_subset_path = project_root / "outputs" / "activations" / "sae_layer_selection_subset_500.csv"
large_probe_subset.to_csv(large_subset_path, index=False)


class ProbeSAE(nn.Module):
    def __init__(self, d_in=768, d_sae=1024):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z


flat_class_ids = []
flat_image_indices = []
for image_index, image_row in large_probe_subset.iterrows():
    for _ in range(196):
        flat_class_ids.append(image_row["class_id"])
        flat_image_indices.append(image_index)


def extract_patch_activations_for_layer(layer_number):
    layer_batches = []

    def save_activation(module, inputs, output):
        layer_batches.append(output.detach().cpu()[:, 1:, :])

    handle = model.encoder.layers[layer_number - 1].register_forward_hook(save_activation)

    for start in range(0, len(large_probe_subset), image_batch_size):
        batch_rows = large_probe_subset.iloc[start:start + image_batch_size]
        batch_images = [
            preprocess(Image.open(image_path).convert("RGB"))
            for image_path in batch_rows["image_path"]
        ]
        batch_x = torch.stack(batch_images, dim=0)

        with torch.no_grad():
            _ = model(batch_x)

    handle.remove()
    return torch.cat(layer_batches, dim=0).float()


def top_patch_feature_stats(z, activation_frequency, max_activation, top_features=20, top_k=10):
    candidate_feature_ids = torch.where(
        (activation_frequency >= 0.005) & (activation_frequency <= 0.5)
    )[0]
    if len(candidate_feature_ids) == 0:
        candidate_feature_ids = torch.arange(z.shape[1])

    candidate_feature_ids = candidate_feature_ids[
        torch.argsort(max_activation[candidate_feature_ids], descending=True)[:top_features]
    ]

    purities = []
    unique_image_counts = []

    for feature_id in candidate_feature_ids.tolist():
        top_indices = torch.topk(z[:, feature_id], k=top_k).indices.tolist()
        top_classes = [flat_class_ids[index] for index in top_indices]
        top_images = [flat_image_indices[index] for index in top_indices]

        dominant_class_count = Counter(top_classes).most_common(1)[0][1]
        purities.append(dominant_class_count / top_k)
        unique_image_counts.append(len(set(top_images)))

    return {
        "avg_top10_class_purity": sum(purities) / len(purities),
        "avg_top10_unique_images": sum(unique_image_counts) / len(unique_image_counts),
    }


result_rows = []

for layer_number in candidate_layers:
    print(f"Processing layer {layer_number}...")
    torch.manual_seed(42)

    patch_tokens = extract_patch_activations_for_layer(layer_number)
    acts = patch_tokens.reshape(-1, 768)
    input_variance = acts.var(dim=0).mean().item()

    sae = ProbeSAE(d_in=768, d_sae=d_sae)
    optimizer = torch.optim.Adam(sae.parameters(), lr=learning_rate)
    dataloader = DataLoader(TensorDataset(acts), batch_size=sae_batch_size, shuffle=True)

    for _ in range(num_epochs):
        for (batch_acts,) in dataloader:
            x_hat, z = sae(batch_acts)
            reconstruction_loss = F.mse_loss(x_hat, batch_acts)
            sparsity_loss = z.abs().mean()
            total_loss = reconstruction_loss + l1_coeff * sparsity_loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

    sae.eval()
    with torch.no_grad():
        x_hat, z = sae(acts)
        reconstruction_mse = F.mse_loss(x_hat, acts).item()
        normalized_reconstruction_mse = reconstruction_mse / max(input_variance, 1e-8)
        active_mask = z > 1e-6
        l0_mean = active_mask.float().sum(dim=1).mean().item()
        activation_frequency = active_mask.float().mean(dim=0)
        max_activation = z.max(dim=0).values
        active_feature_fraction = (activation_frequency > 0.001).float().mean().item()
        dead_feature_fraction = (activation_frequency == 0).float().mean().item()

    feature_stats = top_patch_feature_stats(z, activation_frequency, max_activation)
    feature_diversity_factor = feature_stats["avg_top10_unique_images"] / 10
    sparsity_factor = max(0.0, 1.0 - min(l0_mean / d_sae, 1.0))

    sae_layer_score = (
        feature_stats["avg_top10_class_purity"]
        * active_feature_fraction
        * feature_diversity_factor
        * (0.25 + sparsity_factor)
        / max(normalized_reconstruction_mse, 1e-8)
    )

    result_rows.append({
        "layer": layer_number,
        "activation_shape": tuple(patch_tokens.shape),
        "flattened_shape": tuple(acts.shape),
        "d_sae": d_sae,
        "num_epochs": num_epochs,
        "l1_coeff": l1_coeff,
        "reconstruction_mse": reconstruction_mse,
        "normalized_reconstruction_mse": normalized_reconstruction_mse,
        "l0_mean": l0_mean,
        "active_feature_fraction": active_feature_fraction,
        "dead_feature_fraction": dead_feature_fraction,
        **feature_stats,
        "sae_layer_score": sae_layer_score,
    })

    del patch_tokens, acts, sae, z, x_hat

large_sae_layer_selection = pd.DataFrame(result_rows).sort_values(
    "sae_layer_score",
    ascending=False,
).reset_index(drop=True)

large_output_path = project_root / "outputs" / "feature_rankings" / "sae_layer_selection_probe_500_layers_3_6_9_11.csv"
large_sae_layer_selection.to_csv(large_output_path, index=False)

print(f"Probe subset: {len(large_probe_subset)} images from {large_probe_subset['class_id'].nunique()} classes")
print(f"Saved 500-image SAE layer-selection probe: {large_output_path}")
display(large_sae_layer_selection)




Processing layer 3...
Processing layer 6...
Processing layer 9...
Processing layer 11...
Probe subset: 500 images from 50 classes
Saved 500-image SAE layer-selection probe: /Users/queen/PycharmProjects/VITSAE/outputs/feature_rankings/sae_layer_selection_probe_500_layers_3_6_9_11.csv


,layer,activation_shape,flattened_shape,d_sae,num_epochs,l1_coeff,reconstruction_mse,normalized_reconstruction_mse,l0_mean,active_feature_fraction,dead_feature_fraction,avg_top10_class_purity,avg_top10_unique_images,sae_layer_score
0,3,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.002992,0.043098,620.549866,1.000000,0.000000,0.235,8.95,3.142781
1,9,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.012797,0.031685,828.300598,1.000000,0.000000,0.235,9.05,2.960812
2,6,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.008557,0.033837,694.564758,1.000000,0.000000,0.150,9.90,2.509070
3,11,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.064244,0.100813,508.554077,0.911133,0.000977,0.350,8.40,2.001784


The 500-image SAE layer-selection probe saves two files:

```text
outputs/activations/sae_layer_selection_subset_500.csv
outputs/feature_rankings/sae_layer_selection_probe_500_layers_3_6_9_11.csv
```

Observed 500-image result for the original candidate layers `3, 6, 9, 11`:

| rank | layer | normalized reconstruction MSE | L0 mean | active feature fraction | dead feature fraction | avg top-10 class purity | avg top-10 unique images | SAE layer score |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 3 | 0.0431 | 620.55 | 1.0000 | 0.0000 | 0.235 | 8.95 | 3.1428 |
| 2 | 9 | 0.0317 | 828.30 | 1.0000 | 0.0000 | 0.235 | 9.05 | 2.9608 |
| 3 | 6 | 0.0338 | 694.56 | 1.0000 | 0.0000 | 0.150 | 9.90 | 2.5091 |
| 4 | 11 | 0.1008 | 508.55 | 0.9111 | 0.0010 | 0.350 | 8.40 | 2.0018 |

Interpretation:

- The larger probe changes the picture from the 40-image smoke test. Layer 3 scores highest overall, which means it is very strong for reconstructable local patch features.
- Layer 9 is close behind and has the best case for **semantic local-region SAE**: it remains competitive on SAE metrics while being deeper and more object-aware than layer 3.
- Layer 6 is still a useful middle-layer baseline, but in the 500-image result it is no longer the strongest candidate.
- Layer 11 has the highest top-10 class purity, but worse reconstruction and lower active feature use, so it is not the best main patch-token layer.

Interim decision from this candidate-layer probe:

- If the goal is **low-level local visual features**: choose **layer 3 patch-token SAE**.
- If the goal is **semantic local regions / object-related patch features**: choose **layer 9 patch-token SAE**.
- Keep **layer 6** as a middle-layer baseline.
- Use **layer 11** only as a late/global comparison, not as the main local patch layer.

The next all-layer trend probe is stronger evidence because it checks all 12 layers instead of only these four candidates.


# 3c. All-Layer SAE Trend Probe

Instead of only comparing layers 3, 6, 9, and 11, we can run the same lightweight SAE probe on **all 12 ViT encoder layers**.

The purpose is to find where the metrics change sharply across depth. This is more useful than asking which single layer has the highest score, because early layers often win reconstruction-based metrics while later layers may be more semantic.

This probe uses the same `50 classes x 10 images = 500 images` subset and trains one small patch-token SAE per layer. It records:

- reconstruction quality
- sparsity / L0
- active and dead feature fraction
- top-patch class purity
- top-patch image diversity
- layer-to-layer metric deltas

The goal is to identify an elbow: a layer where the representation changes from low-level local features toward more semantic patch features.


In [7]:
from pathlib import Path
from collections import Counter
from PIL import Image
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights
from IPython.display import display


project_root = Path.cwd()
if not (project_root / "outputs").exists():
    project_root = project_root.parent

all_layers = list(range(1, 13))
num_classes = 50
images_per_class = 10
image_batch_size = 16
sae_batch_size = 2048
d_sae = 1024
num_epochs = 5
l1_coeff = 5e-2
learning_rate = 1e-3

weights = ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

if "model" not in globals():
    model = vit_b_16(weights=weights)
    model.eval()

data_root = project_root / "data" / "val.X"
class_dirs = sorted([path for path in data_root.iterdir() if path.is_dir()])
eligible_class_dirs = [
    class_dir for class_dir in class_dirs
    if len(list(class_dir.glob("*.JPEG"))) >= images_per_class
]
selected_class_dirs = eligible_class_dirs[:num_classes]

subset_rows = []
for class_index, class_dir in enumerate(selected_class_dirs):
    for image_path in sorted(class_dir.glob("*.JPEG"))[:images_per_class]:
        subset_rows.append({
            "image_index": len(subset_rows),
            "class_index": class_index,
            "class_id": class_dir.name,
            "image_path": str(image_path),
        })

all_layer_subset = pd.DataFrame(subset_rows)
all_layer_subset_path = project_root / "outputs" / "activations" / "sae_layer_selection_subset_500.csv"
all_layer_subset.to_csv(all_layer_subset_path, index=False)


class ProbeSAE(nn.Module):
    def __init__(self, d_in=768, d_sae=1024):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_sae)
        self.decoder = nn.Linear(d_sae, d_in)

    def forward(self, x):
        z = F.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z


flat_class_ids = []
flat_image_indices = []
for image_index, image_row in all_layer_subset.iterrows():
    for _ in range(196):
        flat_class_ids.append(image_row["class_id"])
        flat_image_indices.append(image_index)


def extract_patch_activations_for_layer(layer_number):
    layer_batches = []

    def save_activation(module, inputs, output):
        layer_batches.append(output.detach().cpu()[:, 1:, :])

    handle = model.encoder.layers[layer_number - 1].register_forward_hook(save_activation)

    for start in range(0, len(all_layer_subset), image_batch_size):
        batch_rows = all_layer_subset.iloc[start:start + image_batch_size]
        batch_images = [
            preprocess(Image.open(image_path).convert("RGB"))
            for image_path in batch_rows["image_path"]
        ]
        batch_x = torch.stack(batch_images, dim=0)

        with torch.no_grad():
            _ = model(batch_x)

    handle.remove()
    return torch.cat(layer_batches, dim=0).float()


def top_patch_feature_stats(z, activation_frequency, max_activation, top_features=20, top_k=10):
    candidate_feature_ids = torch.where(
        (activation_frequency >= 0.005) & (activation_frequency <= 0.5)
    )[0]
    if len(candidate_feature_ids) == 0:
        candidate_feature_ids = torch.arange(z.shape[1])

    candidate_feature_ids = candidate_feature_ids[
        torch.argsort(max_activation[candidate_feature_ids], descending=True)[:top_features]
    ]

    purities = []
    unique_image_counts = []

    for feature_id in candidate_feature_ids.tolist():
        top_indices = torch.topk(z[:, feature_id], k=top_k).indices.tolist()
        top_classes = [flat_class_ids[index] for index in top_indices]
        top_images = [flat_image_indices[index] for index in top_indices]

        dominant_class_count = Counter(top_classes).most_common(1)[0][1]
        purities.append(dominant_class_count / top_k)
        unique_image_counts.append(len(set(top_images)))

    return {
        "avg_top10_class_purity": sum(purities) / len(purities),
        "avg_top10_unique_images": sum(unique_image_counts) / len(unique_image_counts),
    }


result_rows = []

for layer_number in all_layers:
    print(f"Processing layer {layer_number}...")
    torch.manual_seed(42)

    patch_tokens = extract_patch_activations_for_layer(layer_number)
    acts = patch_tokens.reshape(-1, 768)
    input_variance = acts.var(dim=0).mean().item()

    sae = ProbeSAE(d_in=768, d_sae=d_sae)
    optimizer = torch.optim.Adam(sae.parameters(), lr=learning_rate)
    dataloader = DataLoader(TensorDataset(acts), batch_size=sae_batch_size, shuffle=True)

    for _ in range(num_epochs):
        for (batch_acts,) in dataloader:
            x_hat, z = sae(batch_acts)
            reconstruction_loss = F.mse_loss(x_hat, batch_acts)
            sparsity_loss = z.abs().mean()
            total_loss = reconstruction_loss + l1_coeff * sparsity_loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

    sae.eval()
    with torch.no_grad():
        x_hat, z = sae(acts)
        reconstruction_mse = F.mse_loss(x_hat, acts).item()
        normalized_reconstruction_mse = reconstruction_mse / max(input_variance, 1e-8)
        active_mask = z > 1e-6
        l0_mean = active_mask.float().sum(dim=1).mean().item()
        activation_frequency = active_mask.float().mean(dim=0)
        max_activation = z.max(dim=0).values
        active_feature_fraction = (activation_frequency > 0.001).float().mean().item()
        dead_feature_fraction = (activation_frequency == 0).float().mean().item()

    feature_stats = top_patch_feature_stats(z, activation_frequency, max_activation)
    feature_diversity_factor = feature_stats["avg_top10_unique_images"] / 10
    sparsity_factor = max(0.0, 1.0 - min(l0_mean / d_sae, 1.0))

    sae_layer_score = (
        feature_stats["avg_top10_class_purity"]
        * active_feature_fraction
        * feature_diversity_factor
        * (0.25 + sparsity_factor)
        / max(normalized_reconstruction_mse, 1e-8)
    )

    result_rows.append({
        "layer": layer_number,
        "activation_shape": tuple(patch_tokens.shape),
        "flattened_shape": tuple(acts.shape),
        "d_sae": d_sae,
        "num_epochs": num_epochs,
        "l1_coeff": l1_coeff,
        "reconstruction_mse": reconstruction_mse,
        "normalized_reconstruction_mse": normalized_reconstruction_mse,
        "l0_mean": l0_mean,
        "active_feature_fraction": active_feature_fraction,
        "dead_feature_fraction": dead_feature_fraction,
        **feature_stats,
        "sae_layer_score": sae_layer_score,
    })

    del patch_tokens, acts, sae, z, x_hat

all_layer_sae_results = pd.DataFrame(result_rows).sort_values("layer").reset_index(drop=True)

for column in [
    "normalized_reconstruction_mse",
    "l0_mean",
    "avg_top10_class_purity",
    "avg_top10_unique_images",
    "sae_layer_score",
]:
    all_layer_sae_results[f"delta_{column}"] = all_layer_sae_results[column].diff()

all_layer_output_path = project_root / "outputs" / "feature_rankings" / "sae_layer_selection_probe_500_all12.csv"
all_layer_sae_results.to_csv(all_layer_output_path, index=False)

ranked_all_layer_sae_results = all_layer_sae_results.sort_values(
    "sae_layer_score",
    ascending=False,
).reset_index(drop=True)
ranked_all_layer_output_path = project_root / "outputs" / "feature_rankings" / "sae_layer_selection_probe_500_all12_ranked.csv"
ranked_all_layer_sae_results.to_csv(ranked_all_layer_output_path, index=False)

print(f"Saved all-layer SAE trend probe: {all_layer_output_path}")
print(f"Saved ranked all-layer SAE trend probe: {ranked_all_layer_output_path}")
display(all_layer_sae_results)
display(ranked_all_layer_sae_results)



Processing layer 1...
Processing layer 2...
Processing layer 3...
Processing layer 4...
Processing layer 5...
Processing layer 6...
Processing layer 7...
Processing layer 8...
Processing layer 9...
Processing layer 10...
Processing layer 11...
Processing layer 12...
Saved all-layer SAE trend probe: /Users/queen/PycharmProjects/VITSAE/outputs/feature_rankings/sae_layer_selection_probe_500_all12.csv
Saved ranked all-layer SAE trend probe: /Users/queen/PycharmProjects/VITSAE/outputs/feature_rankings/sae_layer_selection_probe_500_all12_ranked.csv


,layer,activation_shape,flattened_shape,d_sae,num_epochs,l1_coeff,reconstruction_mse,normalized_reconstruction_mse,l0_mean,active_feature_fraction,dead_feature_fraction,avg_top10_class_purity,avg_top10_unique_images,sae_layer_score,delta_normalized_reconstruction_mse,delta_l0_mean,delta_avg_top10_class_purity,delta_avg_top10_unique_images,delta_sae_layer_score
0,1,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.001778,0.030080,499.874847,1.000000,0.000000,0.300000,7.750000,5.888546,NaN,NaN,NaN,NaN,NaN
1,2,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.002339,0.038374,556.100342,1.000000,0.000000,0.345000,7.450000,4.734960,0.008294,56.225494,0.045000,-0.300000,-1.153586
2,3,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.002992,0.043098,620.549866,1.000000,0.000000,0.235000,8.950000,3.142781,0.004724,64.449524,-0.110000,1.500000,-1.592179
3,4,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.003718,0.043384,721.313965,1.000000,0.000000,0.366667,6.666667,3.074085,0.000286,100.764099,0.131667,-2.283333,-0.068696
4,5,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.004679,0.040769,771.409973,1.000000,0.000000,0.300000,8.000000,2.923815,-0.002615,50.096008,-0.066667,1.333333,-0.150269
5,6,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.008557,0.033837,694.564758,1.000000,0.000000,0.150000,9.900000,2.509070,-0.006932,-76.845215,-0.150000,1.900000,-0.414745
6,7,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.010213,0.032750,779.035828,1.000000,0.000000,0.175000,9.650000,2.522656,-0.001087,84.471069,0.025000,-0.250000,0.013586
7,8,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.011715,0.030781,830.532593,1.000000,0.000000,0.195000,9.600000,2.669409,-0.001969,51.496765,0.020000,-0.050000,0.146753
8,9,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.012797,0.031685,828.300598,1.000000,0.000000,0.235000,9.050000,2.960812,0.000904,-2.231995,0.040000,-0.550000,0.291403
9,10,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.023944,0.055422,694.018127,0.987305,0.000000,0.290000,8.450000,2.498109,0.023736,-134.282471,0.055000,-0.600000,-0.462703


,layer,activation_shape,flattened_shape,d_sae,num_epochs,l1_coeff,reconstruction_mse,normalized_reconstruction_mse,l0_mean,active_feature_fraction,dead_feature_fraction,avg_top10_class_purity,avg_top10_unique_images,sae_layer_score,delta_normalized_reconstruction_mse,delta_l0_mean,delta_avg_top10_class_purity,delta_avg_top10_unique_images,delta_sae_layer_score
0,1,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.001778,0.030080,499.874847,1.000000,0.000000,0.300000,7.750000,5.888546,NaN,NaN,NaN,NaN,NaN
1,2,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.002339,0.038374,556.100342,1.000000,0.000000,0.345000,7.450000,4.734960,0.008294,56.225494,0.045000,-0.300000,-1.153586
2,3,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.002992,0.043098,620.549866,1.000000,0.000000,0.235000,8.950000,3.142781,0.004724,64.449524,-0.110000,1.500000,-1.592179
3,4,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.003718,0.043384,721.313965,1.000000,0.000000,0.366667,6.666667,3.074085,0.000286,100.764099,0.131667,-2.283333,-0.068696
4,9,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.012797,0.031685,828.300598,1.000000,0.000000,0.235000,9.050000,2.960812,0.000904,-2.231995,0.040000,-0.550000,0.291403
5,5,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.004679,0.040769,771.409973,1.000000,0.000000,0.300000,8.000000,2.923815,-0.002615,50.096008,-0.066667,1.333333,-0.150269
6,8,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.011715,0.030781,830.532593,1.000000,0.000000,0.195000,9.600000,2.669409,-0.001969,51.496765,0.020000,-0.050000,0.146753
7,7,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.010213,0.032750,779.035828,1.000000,0.000000,0.175000,9.650000,2.522656,-0.001087,84.471069,0.025000,-0.250000,0.013586
8,6,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.008557,0.033837,694.564758,1.000000,0.000000,0.150000,9.900000,2.509070,-0.006932,-76.845215,-0.150000,1.900000,-0.414745
9,10,"(500, 196, 768)","(98000, 768)",1024,5,0.05,0.023944,0.055422,694.018127,0.987305,0.000000,0.290000,8.450000,2.498109,0.023736,-134.282471,0.055000,-0.600000,-0.462703


The all-layer probe saves:

```text
outputs/feature_rankings/sae_layer_selection_probe_500_all12.csv
outputs/feature_rankings/sae_layer_selection_probe_500_all12_ranked.csv
```

Observed ordered trend by layer:

| layer | norm recon MSE | L0 mean | top-10 class purity | unique images | SAE score | delta score |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0.0301 | 499.87 | 0.300 | 7.75 | 5.8885 | N/A |
| 2 | 0.0384 | 556.10 | 0.345 | 7.45 | 4.7350 | -1.1536 |
| 3 | 0.0431 | 620.55 | 0.235 | 8.95 | 3.1428 | -1.5922 |
| 4 | 0.0434 | 721.31 | 0.367 | 6.67 | 3.0741 | -0.0687 |
| 5 | 0.0408 | 771.41 | 0.300 | 8.00 | 2.9238 | -0.1503 |
| 6 | 0.0338 | 694.56 | 0.150 | 9.90 | 2.5091 | -0.4147 |
| 7 | 0.0328 | 779.04 | 0.175 | 9.65 | 2.5227 | +0.0136 |
| 8 | 0.0308 | 830.53 | 0.195 | 9.60 | 2.6694 | +0.1468 |
| 9 | 0.0317 | 828.30 | 0.235 | 9.05 | 2.9608 | +0.2914 |
| 10 | 0.0554 | 694.02 | 0.290 | 8.45 | 2.4981 | -0.4627 |
| 11 | 0.1008 | 508.55 | 0.350 | 8.40 | 2.0018 | -0.4963 |
| 12 | 0.0907 | 541.56 | 0.285 | 8.80 | 1.8649 | -0.1369 |

Visible changes:

- **Layers 1-4** have the highest SAE scores, but this mostly means early patch activations are easier to reconstruct and strongly local. This is good for low-level local visual features, not necessarily semantic object regions.
- **Layers 6-9** form a middle-to-late region where reconstruction stays good and unique-image diversity is high. The score starts rising again from layer 7 to layer 9.
- **Layer 9** is the strongest late-layer candidate: it is the best layer after the early low-level block and appears right before the sharp degradation at layer 10/11.
- **Layers 10-11** show a clear transition: normalized reconstruction error jumps from `0.0317` at layer 9 to `0.0554` at layer 10 and `0.1008` at layer 11. This suggests late patch activations become harder for this small SAE and likely more globally/classification mixed.

Layer choice from the all-layer SAE trend:

- For **low-level local visual features**: choose **layer 3 or 4**.
- For **semantic local regions / object-related patch features**: choose **layer 9**.
- For a middle baseline: keep **layer 6**.
- Avoid **layer 10/11/12** as the main patch-token SAE layer unless the goal is specifically late/global patch behavior.

This resolves the layer-3 issue: layer 3 is numerically strong because it is low-level and easy to reconstruct, but layer 9 is the better project choice if the target is semantic local-region interpretation.


# 4. Decide Which Dataset to Use

The dataset should match the pretrained ViT-B/16 model. Since this notebook uses `ViT_B_16_Weights.IMAGENET1K_V1`, ImageNet-style images are the natural choice.

The main question is not whether to use ImageNet, but **which ImageNet scale** to use. Full ImageNet-1K train is too large for the current exploratory SAE workflow, while very tiny subsets are too small for stable feature interpretation.

Dataset options:

| dataset option | classes | images | approximate size | advantages | limitations | decision |
|---|---:|---:|---:|---|---|---|
| ImageNet-1K train | 1,000 | ~1.28M | ~150GB | full training distribution; largest coverage | too large; slow activation extraction; high activation storage cost | not now |
| ImageNet-1K validation | 1,000 | 50,000 | much smaller than train | covers all 1,000 classes; standard evaluation split | still 10x larger than local ImageNet-100 validation | future scaling option |
| ImageNet-100 validation | 100 | 5,000 | ~711MB locally | manageable; diverse; aligned with ImageNet-pretrained ViT | fewer classes than full ImageNet-1K | use now |
| 500-image subset | 50 | 500 | small | fast enough for SAE layer-selection probes | not enough for final feature claims | probe only |
| 40-image subset | 10 | 40 | tiny | smoke test for code correctness | too small for real layer choice | smoke test only |

For this project, the current local dataset is an **ImageNet-100 validation subset**:

```text
100 classes x 50 validation images per class = 5,000 images
```

This is large enough to test SAE behavior across many categories, but small enough to run locally. It is a good middle ground between tiny toy subsets and full ImageNet-1K.

Dataset decision:

- Use **ImageNet-100 validation set** as the main dataset for this project stage.
- Use smaller subsets such as `50 classes x 10 images = 500 images` for fast layer-selection probes.
- Use the full 5,000-image ImageNet-100 validation subset for more serious SAE training and feature interpretation.
- Consider **ImageNet-1K validation** later if we need stronger class coverage.
- Do not use full **ImageNet-1K train** yet because it is too large for the current exploratory workflow.


In [8]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

data_root = project_root / "data" / "val.X"
class_dirs = sorted([path for path in data_root.iterdir() if path.is_dir()])

rows = []
for class_dir in class_dirs:
    image_paths = sorted(class_dir.glob("*.JPEG"))
    rows.append({
        "class_id": class_dir.name,
        "num_images": len(image_paths),
        "example_image_path": str(image_paths[0]) if image_paths else None,
    })

dataset_summary = pd.DataFrame(rows)
num_classes = len(dataset_summary)
num_images = int(dataset_summary["num_images"].sum())
min_images_per_class = int(dataset_summary["num_images"].min())
max_images_per_class = int(dataset_summary["num_images"].max())

print(f"Dataset root: {data_root}")
print(f"Number of classes: {num_classes}")
print(f"Number of images: {num_images}")
print(f"Images per class: min={min_images_per_class}, max={max_images_per_class}")

dataset_summary_path = project_root / "outputs" / "activations" / "imagenet100_validation_dataset_summary.csv"
dataset_summary.to_csv(dataset_summary_path, index=False)
print(f"Saved dataset summary: {dataset_summary_path}")

display(dataset_summary.head())



Dataset root: /Users/queen/PycharmProjects/VITSAE/data/val.X
Number of classes: 100
Number of images: 5000
Images per class: min=50, max=50
Saved dataset summary: /Users/queen/PycharmProjects/VITSAE/outputs/activations/imagenet100_validation_dataset_summary.csv


,class_id,num_images,example_image_path
0,n01440764,50,/Users/queen/PycharmProjects/VITSAE/data/val.X...
1,n01443537,50,/Users/queen/PycharmProjects/VITSAE/data/val.X...
2,n01484850,50,/Users/queen/PycharmProjects/VITSAE/data/val.X...
3,n01491361,50,/Users/queen/PycharmProjects/VITSAE/data/val.X...
4,n01494475,50,/Users/queen/PycharmProjects/VITSAE/data/val.X...


How to use subsets in this project:

| purpose | subset size | reason |
|---|---:|---|
| smoke test | 40 images | fast check that hooks, SAE training, and metrics work |
| layer-selection probe | 500 images | enough diversity to compare candidate layers without full training cost |
| main exploratory SAE | 5,000 images | full local ImageNet-100 validation subset for more stable feature rankings |
| future scaling | 50,000 images | ImageNet-1K validation if we need all 1,000 classes |
| full ImageNet-1K train | not now | too large for current local exploratory workflow |

Final dataset choice: **ImageNet-100 validation set now; ImageNet-1K validation later if needed; not ImageNet-1K train at this stage**.

This keeps the data aligned with the pretrained ImageNet ViT while keeping storage and activation extraction manageable.


# 5. Check Image Preprocessing

Yes, preprocessing is required before feeding images into ViT-B/16. The model expects a normalized tensor with shape `[batch, 3, 224, 224]`.

But we should **not manually reimplement** the preprocessing. Since we use torchvision pretrained weights, the safest choice is to use the transform attached to the weights:

```python
weights = ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
```

This official transform handles the required preprocessing for the pretrained ViT-B/16 weights, including:

- resize
- center crop to `224 x 224`
- convert image to tensor
- normalize using the ImageNet mean and standard deviation expected by the pretrained model

So the answer is: **preprocessing is needed, but it should mainly come from the model weights, not from hand-written image processing code**.


In [9]:
from pathlib import Path
from PIL import Image
from torchvision.models import ViT_B_16_Weights


project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

weights = ViT_B_16_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

example_image_path = sorted((project_root / "data" / "val.X").glob("*/*.JPEG"))[0]
image = Image.open(example_image_path).convert("RGB")
preprocessed_image = preprocess(image)
batched_image = preprocessed_image.unsqueeze(0)

print(f"Example image path: {example_image_path}")
print(f"Original PIL image size: {image.size}")
print(f"Preprocess transform: {preprocess}")
print(f"Preprocessed image shape: {tuple(preprocessed_image.shape)}")
print(f"Batched image shape: {tuple(batched_image.shape)}")
print(f"Tensor dtype: {preprocessed_image.dtype}")
print(f"Tensor min value: {preprocessed_image.min().item():.4f}")
print(f"Tensor max value: {preprocessed_image.max().item():.4f}")

assert tuple(preprocessed_image.shape) == (3, 224, 224)
assert tuple(batched_image.shape) == (1, 3, 224, 224)



Example image path: /Users/queen/PycharmProjects/VITSAE/data/val.X/n01440764/ILSVRC2012_val_00000293.JPEG
Original PIL image size: (500, 375)
Preprocess transform: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)
Preprocessed image shape: (3, 224, 224)
Batched image shape: (1, 3, 224, 224)
Tensor dtype: torch.float32
Tensor min value: -1.6727
Tensor max value: 2.5354


Implementation decision:

- Use `weights.transforms()` everywhere before passing images into ViT.
- Do not manually hard-code resize, crop, mean, or std unless we have a specific reason.
- Save activations only after this preprocessing and the ViT forward pass.

This matters because SAE features depend on the ViT activations. If preprocessing is inconsistent with the pretrained weights, the activations can shift and the SAE interpretation becomes unreliable.


# Final Summary

The notebook answers the requested foundation questions and gives a practical project setup.

| question | answer |
|---|---|
| How is SAE implemented? | A sparse autoencoder trained on frozen ViT activations: encoder, ReLU sparse code, decoder, MSE + L1 loss. |
| What is ViT-B/16? | A pretrained ImageNet Vision Transformer with 16x16 patches, 12 encoder blocks, 12 attention heads, 768 hidden dimension, and 1000-class logits. |
| What layer should we use? | For semantic local patch features, use **layer 9 patch tokens**. Use layer 3/4 for low-level features, layer 6 as baseline, layer 11 as late comparison. |
| What dataset should we use? | Use **ImageNet-100 validation** now: 5,000 images from 100 classes. Use 500-image subsets for probes. Avoid full ImageNet-1K train for now. |
| Is preprocessing needed? | Yes. Use `ViT_B_16_Weights.IMAGENET1K_V1.transforms()` so preprocessing matches the pretrained model. |

Main project recommendation: train the next serious patch-token SAE on **layer 9** using the **ImageNet-100 validation set**, with official ViT-B/16 preprocessing from `weights.transforms()`.
